# Day 3 · 문지기 `check_order`

**여는 법:** `런타임 → 모두 실행`. 세션이 끊기면 첫 Cell부터.
**기대 결과:** 기록 12건 · 통과 10건 · 거부 2건 · 통과 합계 112,250원. 거부 이유는 `없는 메뉴: 녹차라떼`, `필수 Key 없음: is_student`.
**한계:** 문지기의 2·3번 조건을 채우기 전에는 `놓침 1건`이 나옵니다. 놓침이 0이 되도록 조건을 채웁니다.

In [ ]:
# 첫 Cell · 세션이 새로 시작될 때마다 이 Cell부터 실행합니다.
import os, sys
if not os.path.isdir("jnu-llmops-precourse-day2"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git
sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료")

## 1. 주문 기록 12건을 파일에서 읽습니다

In [ ]:
import json
from catalog import MENU
from order import Order
from pricing import calculate_bill

with open("jnu-llmops-precourse-day3/data/orders.json", encoding="utf-8") as f:
    records = json.load(f)

print(len(records))
print(records[0]["items"][0]["menu_name"])

## 2. 문지기 없이 그대로 돌리면 7번째에서 멈춥니다
아래 Cell은 일부러 멈추는 모습을 보여 줍니다. 오류를 붙잡아 출력만 하므로 `모두 실행`은 계속됩니다.

In [ ]:
import traceback
try:
    for record in records:
        order = Order()
        for item in record["items"]:
            order.add(item["menu_name"], item["quantity"], MENU[item["menu_name"]])
        bill = calculate_bill(order, record["is_student"])
        print(record["order_id"], bill.total)
except Exception:
    traceback.print_exc()

## 3. 문지기 — 세 조건 중 둘을 채웁니다
`check_order`는 기록 한 건을 받아 `(True, "")` 또는 `(False, 이유)`를 돌려줍니다. 1번은 완성돼 있습니다. 2·3번의 `False` 자리를 조건으로 바꿉니다.

In [ ]:
REQUIRED_KEYS = ("order_id", "items", "is_student")

def check_order(record):
    for key in REQUIRED_KEYS:                       # 1. 필수 Key  (완성 예)
        if key not in record:
            return False, f"필수 Key 없음: {key}"
    for item in record["items"]:
        if False:                                   # 2. 허용 메뉴  ← 이 조건을 채웁니다
            return False, f"없는 메뉴: {item['menu_name']}"
        quantity = item["quantity"]
        if False:                                   # 3. 수량 범위  ← 이 조건을 채웁니다
            return False, f"수량 범위 밖: {quantity!r}"
    return True, ""


In [ ]:
print(check_order(records[0]))    # 기대 (True, '')
print(check_order(records[6]))    # 기대 (False, '없는 메뉴: 녹차라떼')
print(check_order(records[10]))   # 기대 (False, '필수 Key 없음: is_student')

## 4. 통과와 거부로 나눠 기대표와 대조합니다

In [ ]:
accepted, rejected, missed = [], [], []
for record in records:
    ok, reason = check_order(record)
    if not ok:
        rejected.append({"order_id": record["order_id"], "reason": reason})
        continue
    try:
        order = Order()
        for item in record["items"]:
            order.add(item["menu_name"], item["quantity"], MENU[item["menu_name"]])
        bill = calculate_bill(order, record["is_student"])
        accepted.append({"order_id": record["order_id"], "total": bill.total})
    except Exception as e:                     # 문지기가 놓친 것은 여기서 드러납니다
        missed.append({"order_id": record["order_id"], "error": f"{type(e).__name__}: {e}"})

print("통과", len(accepted), "건 · 거부", len(rejected), "건 · 놓침", len(missed), "건 · 통과 합계", sum(a["total"] for a in accepted), "원")
for r in rejected:
    print("거부", r["order_id"], "-", r["reason"])
for m in missed:
    print("놓침", m["order_id"], "-", m["error"], "← 문지기가 먼저 거부했어야 합니다")

In [ ]:
with open("jnu-llmops-precourse-day3/data/expected_day3.json", encoding="utf-8") as f:
    expected = json.load(f)
print("기대:", expected["accepted"], "건 통과 ·", expected["rejected"], "건 거부 · 합계", expected["accepted_total"], "원")
print("관찰:", len(accepted), "건 통과 ·", len(rejected), "건 거부 · 합계", sum(a["total"] for a in accepted), "원")

## 5. 결과를 파일로 남깁니다 — 왼쪽 파일 탭에서 확인

In [ ]:
with open("orders_result.json", "w", encoding="utf-8") as f:
    json.dump({"accepted": accepted, "rejected": rejected}, f, ensure_ascii=False, indent=2)
print("저장:", "orders_result.json")

## 확장 (빠른 학생) — 깨진 기록을 하나 더 넣어 봅니다
수량 0인 기록과 수량이 `"두"`인 기록을 만들어 `check_order`에 넣고, 어느 조건이 거부하는지 예측한 뒤 확인합니다. 정상 10건의 합계 112,250원은 그대로여야 합니다.

In [ ]:
extra = [
    {"order_id": "X01", "items": [{"menu_name": "카페라떼", "quantity": 0}], "is_student": False},
    {"order_id": "X02", "items": [{"menu_name": "카페라떼", "quantity": "두"}], "is_student": False},
]
for record in extra:
    print(record["order_id"], check_order(record))

## 인계 — 세 문장을 적습니다
- 입력:
- 결과:
- 변경: